# Customer Churn Prediction - Inference Pipeline
This notebook demonstrates loading the trained ANN model (`model.h5`), encoders, and scaler to predict customer churn probability.

In [ ]:
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf

In [ ]:
# Load trained model and preprocessing artifacts
model = tf.keras.models.load_model('model.h5')

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('onehot_encoder_geo.pkl', 'rb') as f:
    onehot_encoder_geo = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print("Artifacts successfully loaded.")

In [ ]:
# Example input data for prediction
sample_input = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
}

# 1. Encode Gender
gender_encoded = label_encoder_gender.transform([sample_input['Gender']])[0]

# 2. Encode Geography
geo_df = pd.DataFrame([[sample_input['Geography']]], columns=['Geography'])
geo_encoded = onehot_encoder_geo.transform(geo_df).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

# 3. Create DataFrame
input_df = pd.DataFrame({
    'CreditScore': [sample_input['CreditScore']],
    'Gender': [gender_encoded],
    'Age': [sample_input['Age']],
    'Tenure': [sample_input['Tenure']],
    'Balance': [sample_input['Balance']],
    'NumOfProducts': [sample_input['NumOfProducts']],
    'HasCrCard': [sample_input['HasCrCard']],
    'IsActiveMember': [sample_input['IsActiveMember']],
    'EstimatedSalary': [sample_input['EstimatedSalary']]
})

full_input_df = pd.concat([input_df, geo_encoded_df], axis=1)
full_input_df = full_input_df[scaler.feature_names_in_]
full_input_df

In [ ]:
# 4. Scale features & Predict
scaled_data = scaler.transform(full_input_df)
prediction = model.predict(scaled_data)
prediction_proba = prediction[0][0]

print(f"Churn Probability: {prediction_proba:.4f} ({prediction_proba * 100:.2f}%)")
if prediction_proba > 0.5:
    print("Result: Customer is likely to churn / exit.")
else:
    print("Result: Customer is not likely to churn.")